# OpenAI Cookbook Comprehensive Guide
## Advanced GPT-5 Implementation & Production Deployment

This notebook provides a comprehensive guide to implementing OpenAI's latest capabilities, covering GPT-5 features, evaluation frameworks, multimodal applications, and production deployment patterns.

**Key Topics Covered:**
- GPT-5 Advanced Features (Verbosity, Reasoning, CFG)
- API Integration Patterns (Responses vs Chat Completions)
- Evaluation Framework Setup with Evals API
- Multimodal Applications (Text, Audio, Vision)
- Agent Development with OpenAI Agents SDK
- Production Deployment Best Practices
- Enterprise Integration Examples

**Prerequisites:**
- OpenAI API key with GPT-5 access
- Python 3.12+ environment
- Basic familiarity with API concepts

**EQ12 Project Integration:**
This guide is tailored for the EQ12 sports betting automation project, providing practical examples that can be directly applied to financial data processing, real-time analysis, and automated decision systems.

## Section 1: OpenAI Cookbook Overview and Navigation

The OpenAI Cookbook provides extensive resources for implementing AI solutions. Let's start by setting up our environment and exploring the key patterns we'll implement.

In [ ]:
# Essential Library Imports for OpenAI Cookbook Implementation
import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any

# Visualization
# OpenAI SDK and related tools
import openai

# Data processing and analysis
# Environment setup
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI

# Pydantic for structured outputs
from pydantic import BaseModel, Field

load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
async_client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Environment setup complete")
print(f"📊 OpenAI SDK version: {openai.__version__}")
print(f"🔑 API key configured: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

# Verify API connection
try:
    models = client.models.list()
    available_models = [model.id for model in models.data if "gpt" in model.id]
    print(f"🤖 Available GPT models: {len([m for m in available_models if 'gpt' in m])}")
    print(f"🚀 GPT-5 available: {'Yes' if any('gpt-5' in m for m in available_models) else 'No'}")
except Exception as e:
    print(f"❌ API connection failed: {e}")

In [ ]:
# EQ12 Project Configuration and Utilities
class EQ12Config:
    """Configuration class for EQ12 sports betting automation project"""

    def __init__(self):
        self.project_root = Path("C:/EQ12")
        self.logs_dir = self.project_root / "logs"
        self.data_dir = self.project_root / "data"
        self.configs_dir = self.project_root / "configs"

        # Create directories if they don't exist
        for directory in [self.logs_dir, self.data_dir, self.configs_dir]:
            directory.mkdir(parents=True, exist_ok=True)

    def log_operation(self, operation: str, data: dict[Any, Any]):
        """Log operations with UTC timestamps as required by EQ12 standards"""
        timestamp = datetime.utcnow().isoformat()
        log_entry = {"timestamp": timestamp, "operation": operation, "data": data}

        log_file = self.logs_dir / f"openai_operations_{datetime.utcnow().strftime('%Y%m%d')}.json"

        # Append to daily log file
        if log_file.exists():
            with open(log_file) as f:
                logs = json.load(f)
        else:
            logs = []

        logs.append(log_entry)

        with open(log_file, "w") as f:
            json.dump(logs, f, indent=2)

        return log_entry


# Initialize EQ12 configuration
eq12_config = EQ12Config()
print(f"📁 EQ12 project initialized at: {eq12_config.project_root}")
print(f"📝 Logs directory: {eq12_config.logs_dir}")

# Test logging functionality
eq12_config.log_operation(
    "notebook_initialization",
    {"environment": "jupyter", "openai_version": openai.__version__, "python_version": "3.12+"},
)

print("✅ EQ12 configuration and logging system ready")

## Section 2: GPT-5 Advanced Features Implementation

GPT-5 introduces several groundbreaking features that revolutionize AI application development:

### Key New Features:
1. **Verbosity Parameter**: Control response length and detail level
2. **Reasoning Effort**: Adjust thinking depth from minimal to high
3. **Context-Free Grammar (CFG)**: Enforce strict output formatting
4. **Freeform Function Calling**: Send raw text payloads to custom tools
5. **Minimal Reasoning**: Ultra-fast responses for deterministic tasks

Let's implement each feature with practical examples relevant to EQ12's sports betting automation needs.

In [ ]:
# GPT-5 Verbosity Parameter Implementation
def demonstrate_verbosity_control():
    """Demonstrate GPT-5's verbosity parameter for sports betting analysis"""

    prompt = "Analyze the key factors affecting sports betting odds for NFL games"

    verbosity_levels = ["low", "medium", "high"]
    results = {}

    for verbosity in verbosity_levels:
        try:
            response = client.responses.create(
                model="gpt-5-mini",  # Using gpt-5-mini as it's more accessible
                input=prompt,
                text={"verbosity": verbosity},
            )

            # Extract output text
            output_text = ""
            for item in response.output:
                if hasattr(item, "content"):
                    for content in item.content:
                        if hasattr(content, "text"):
                            output_text += content.text

            results[verbosity] = {
                "text": output_text,
                "tokens": response.usage.output_tokens if response.usage else 0,
            }

            print(f"\n{'=' * 50}")
            print(f"VERBOSITY: {verbosity.upper()}")
            print(f"Tokens: {results[verbosity]['tokens']}")
            print(f"{'=' * 50}")
            print(
                results[verbosity]["text"][:500] + "..."
                if len(results[verbosity]["text"]) > 500
                else results[verbosity]["text"]
            )

        except Exception as e:
            print(f"❌ Error with {verbosity} verbosity: {e}")
            results[verbosity] = {"error": str(e)}

    # Log results for EQ12
    eq12_config.log_operation(
        "gpt5_verbosity_demo",
        {"prompt": prompt, "results": results, "feature": "verbosity_control"},
    )

    return results


# Execute verbosity demonstration
print("🔄 Testing GPT-5 Verbosity Control...")
verbosity_results = demonstrate_verbosity_control()

In [ ]:
# GPT-5 Reasoning Effort Implementation
def demonstrate_reasoning_effort():
    """Demonstrate different reasoning effort levels for complex betting analysis"""

    complex_prompt = """
    Given these NFL game conditions:
    - Team A: 8-3 record, home field advantage, star QB healthy
    - Team B: 6-5 record, away, backup QB starting
    - Weather: Light rain expected
    - Historical H2H: Team A won last 3 meetings
    - Betting line: Team A -7.5, Over/Under 45.5

    Provide a comprehensive betting recommendation with confidence levels.
    """

    reasoning_levels = ["minimal", "medium", "high"]
    results = {}

    for effort in reasoning_levels:
        try:
            start_time = time.time()

            response = client.responses.create(
                model="gpt-5-mini",
                input=complex_prompt,
                reasoning={"effort": effort},
                text={"verbosity": "medium"},
            )

            end_time = time.time()

            # Extract output
            output_text = ""
            for item in response.output:
                if hasattr(item, "content"):
                    for content in item.content:
                        if hasattr(content, "text"):
                            output_text += content.text

            results[effort] = {
                "text": output_text,
                "tokens": response.usage.total_tokens if response.usage else 0,
                "latency": round(end_time - start_time, 2),
            }

            print(f"\n{'=' * 60}")
            print(f"REASONING EFFORT: {effort.upper()}")
            print(f"Latency: {results[effort]['latency']}s | Tokens: {results[effort]['tokens']}")
            print(f"{'=' * 60}")
            print(
                results[effort]["text"][:600] + "..."
                if len(results[effort]["text"]) > 600
                else results[effort]["text"]
            )

        except Exception as e:
            print(f"❌ Error with {effort} reasoning: {e}")
            results[effort] = {"error": str(e)}

    # Log results
    eq12_config.log_operation(
        "gpt5_reasoning_effort_demo",
        {"prompt": complex_prompt, "results": results, "feature": "reasoning_effort"},
    )

    return results


# Execute reasoning effort demonstration
print("🧠 Testing GPT-5 Reasoning Effort Levels...")
reasoning_results = demonstrate_reasoning_effort()

In [ ]:
# Context-Free Grammar (CFG) Implementation
def demonstrate_context_free_grammar():
    """Demonstrate CFG for structured betting data output"""

    # Define Lark grammar for betting odds format
    betting_odds_grammar = """
        // Betting odds grammar for structured output
        start: bet_analysis

        bet_analysis: team_analysis "\\n" odds_analysis "\\n" recommendation

        team_analysis: "TEAM_A:" SP team_stats "\\n" "TEAM_B:" SP team_stats
        team_stats: IDENTIFIER SP record SP form_rating

        odds_analysis: "SPREAD:" SP spread_value "\\n" "TOTAL:" SP total_value
        spread_value: TEAM SP NUMBER
        total_value: "O/U" SP NUMBER

        recommendation: "PICK:" SP bet_choice "\\n" "CONFIDENCE:" SP confidence_level
        bet_choice: TEAM | "OVER" | "UNDER"
        confidence_level: "HIGH" | "MEDIUM" | "LOW"

        // Terminals
        SP: " "
        TEAM: /[A-Z]{3}/
        IDENTIFIER: /[A-Za-z_][A-Za-z0-9_]*/
        NUMBER: /[0-9]+(\\.[0-9]+)?/
        record: /[0-9]+-[0-9]+/
        form_rating: /[0-9]\\.[0-9]/
    """

    try:
        response = client.responses.create(
            model="gpt-5-mini",
            input="Generate betting analysis for Patriots vs Bills game with structured odds output",
            tools=[
                {
                    "type": "custom",
                    "name": "betting_odds_formatter",
                    "description": "Formats betting analysis in structured format for EQ12 system consumption",
                    "format": {
                        "type": "grammar",
                        "syntax": "lark",
                        "definition": betting_odds_grammar,
                    },
                }
            ],
            parallel_tool_calls=False,
        )

        # Extract tool call result
        if len(response.output) > 1 and hasattr(response.output[1], "input"):
            structured_output = response.output[1].input
            print("📊 Structured Betting Analysis Generated:")
            print("=" * 50)
            print(structured_output)
            print("=" * 50)

            # Log for EQ12
            eq12_config.log_operation(
                "gpt5_cfg_demo",
                {
                    "grammar_type": "betting_odds",
                    "structured_output": structured_output,
                    "feature": "context_free_grammar",
                },
            )

            return structured_output
        print("⚠️ No structured output generated")
        return None

    except Exception as e:
        print(f"❌ CFG Error: {e}")
        return None


# Execute CFG demonstration
print("📋 Testing Context-Free Grammar for Structured Output...")
cfg_result = demonstrate_context_free_grammar()

## Section 3: API Integration Patterns

This section compares the Responses API with Chat Completions and demonstrates advanced integration patterns including function calling, tool integration, and structured output generation.

In [ ]:
# Responses API vs Chat Completions Comparison
async def compare_api_patterns():
    """Compare Responses API with Chat Completions for EQ12 use cases"""

    prompt = "Analyze current NFL betting trends and provide three key insights"

    results = {}

    # 1. Chat Completions API (Traditional)
    try:
        start_time = time.time()

        chat_response = client.chat.completions.create(
            model="gpt-4o-mini",  # Using available model
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert sports betting analyst for EQ12 automation system.",
                },
                {"role": "user", "content": prompt},
            ],
            max_tokens=500,
        )

        chat_time = time.time() - start_time

        results["chat_completions"] = {
            "response": chat_response.choices[0].message.content,
            "tokens": chat_response.usage.total_tokens,
            "latency": round(chat_time, 2),
            "model": chat_response.model,
        }

        print("💬 Chat Completions API Result:")
        print(f"Model: {results['chat_completions']['model']}")
        print(
            f"Tokens: {results['chat_completions']['tokens']}, Latency: {results['chat_completions']['latency']}s"
        )
        print(f"Response: {results['chat_completions']['response'][:300]}...")

    except Exception as e:
        print(f"❌ Chat Completions Error: {e}")
        results["chat_completions"] = {"error": str(e)}

    # 2. Responses API (GPT-5 Optimized)
    try:
        start_time = time.time()

        responses_response = client.responses.create(
            model="gpt-4o-mini",  # Will use gpt-5-mini when available
            input=[
                {
                    "role": "developer",
                    "content": [
                        {
                            "type": "input_text",
                            "text": "You are an expert sports betting analyst for EQ12 automation system.",
                        }
                    ],
                },
                {"role": "user", "content": [{"type": "input_text", "text": prompt}]},
            ],
            text={"verbosity": "medium"},
            reasoning={"effort": "medium"},
        )

        responses_time = time.time() - start_time

        # Extract output
        output_text = ""
        for item in responses_response.output:
            if hasattr(item, "content"):
                for content in item.content:
                    if hasattr(content, "text"):
                        output_text += content.text

        results["responses_api"] = {
            "response": output_text,
            "tokens": responses_response.usage.total_tokens if responses_response.usage else 0,
            "latency": round(responses_time, 2),
            "model": (
                responses_response.model if hasattr(responses_response, "model") else "gpt-4o-mini"
            ),
        }

        print("\n🚀 Responses API Result:")
        print(
            f"Tokens: {results['responses_api']['tokens']}, Latency: {results['responses_api']['latency']}s"
        )
        print(f"Response: {results['responses_api']['response'][:300]}...")

    except Exception as e:
        print(f"❌ Responses API Error: {e}")
        results["responses_api"] = {"error": str(e)}

    # Compare and log results
    if "error" not in results["chat_completions"] and "error" not in results["responses_api"]:
        print("\n📊 COMPARISON:")
        print(
            f"Latency Improvement: {((results['chat_completions']['latency'] - results['responses_api']['latency']) / results['chat_completions']['latency'] * 100):.1f}%"
        )
        print(
            f"Token Efficiency: Chat={results['chat_completions']['tokens']}, Responses={results['responses_api']['tokens']}"
        )

    eq12_config.log_operation(
        "api_comparison",
        {
            "prompt": prompt,
            "results": results,
            "comparison_metrics": ["latency", "tokens", "quality"],
        },
    )

    return results


# Execute API comparison
print("⚖️ Comparing API Patterns...")
api_comparison = await compare_api_patterns()

In [ ]:
# Advanced Function Calling and Tool Integration
class EQ12BettingTools:
    """EQ12-specific tools for sports betting automation"""

    def __init__(self):
        self.tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_live_odds",
                    "description": "Retrieve live betting odds for specified games",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "sport": {"type": "string", "enum": ["NFL", "NBA", "MLB"]},
                            "teams": {"type": "array", "items": {"type": "string"}},
                            "bet_types": {
                                "type": "array",
                                "items": {
                                    "type": "string",
                                    "enum": ["spread", "moneyline", "total"],
                                },
                            },
                        },
                        "required": ["sport", "teams"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "calculate_expected_value",
                    "description": "Calculate expected value for betting opportunities",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "odds": {"type": "number"},
                            "probability": {"type": "number", "minimum": 0, "maximum": 1},
                            "stake": {"type": "number", "minimum": 0},
                        },
                        "required": ["odds", "probability", "stake"],
                    },
                },
            },
        ]

    def execute_tool(self, tool_name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        """Execute tool functions (simulated for demo)"""

        if tool_name == "get_live_odds":
            # Simulate API call to odds provider
            return {
                "success": True,
                "data": {
                    "game": f"{arguments['teams'][0]} vs {arguments['teams'][1]}",
                    "odds": {
                        "spread": {"line": -3.5, "odds": -110},
                        "moneyline": {"favorite": -150, "underdog": +130},
                        "total": {"points": 47.5, "over": -105, "under": -115},
                    },
                    "timestamp": datetime.utcnow().isoformat(),
                },
            }

        if tool_name == "calculate_expected_value":
            # Calculate EV = (probability × win_amount) - ((1 - probability) × stake)
            odds = arguments["odds"]
            prob = arguments["probability"]
            stake = arguments["stake"]

            if odds > 0:  # American odds positive
                win_amount = stake * (odds / 100)
            else:  # American odds negative
                win_amount = stake * (100 / abs(odds))

            expected_value = (prob * win_amount) - ((1 - prob) * stake)

            return {
                "success": True,
                "data": {
                    "expected_value": round(expected_value, 2),
                    "roi_percentage": round((expected_value / stake) * 100, 2),
                    "recommendation": "BET" if expected_value > 0 else "PASS",
                },
            }

        return {"success": False, "error": "Unknown tool"}


def demonstrate_function_calling():
    """Demonstrate function calling with EQ12 betting tools"""

    betting_tools = EQ12BettingTools()

    prompt = """
    I need to analyze a betting opportunity for the upcoming Patriots vs Bills game.
    Please get the live odds and calculate the expected value if I believe Patriots have a 60% chance to cover the spread.
    Use a $100 stake for the calculation.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are an EQ12 betting analysis assistant with access to live odds and EV calculation tools.",
                },
                {"role": "user", "content": prompt},
            ],
            tools=betting_tools.tools,
            tool_choice="auto",
        )

        message = response.choices[0].message

        if message.tool_calls:
            print("🔧 Function Calls Detected:")

            for tool_call in message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                print(f"\n📞 Calling: {function_name}")
                print(f"Arguments: {function_args}")

                # Execute tool
                result = betting_tools.execute_tool(function_name, function_args)
                print(f"Result: {result}")

                # Log tool execution
                eq12_config.log_operation(
                    "tool_execution",
                    {"tool": function_name, "arguments": function_args, "result": result},
                )

        print(f"\n🤖 Assistant Response: {message.content}")

        return response

    except Exception as e:
        print(f"❌ Function calling error: {e}")
        return None


# Execute function calling demonstration
print("🔨 Testing Function Calling with EQ12 Tools...")
function_calling_result = demonstrate_function_calling()

## Section 4: Evaluation Framework Setup

Building robust evaluation systems is crucial for production AI applications. This section implements the Evals API with custom graders, business metrics tracking, and continuous improvement pipelines based on the receipt parsing example from the cookbook.

In [ ]:
# EQ12 Betting Prediction Evaluation System
class BettingPrediction(BaseModel):
    """Structured betting prediction model"""

    game_id: str
    team_home: str
    team_away: str
    prediction_type: str  # "spread", "moneyline", "total"
    predicted_value: float
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str
    timestamp: str


class BettingEvaluationRecord(BaseModel):
    """Evaluation record for betting predictions"""

    prediction_id: str
    correct_prediction: BettingPrediction
    model_prediction: BettingPrediction
    actual_outcome: dict[str, Any]
    profit_loss: float


class EQ12EvaluationFramework:
    """Evaluation framework for EQ12 betting predictions"""

    def __init__(self):
        self.graders = self.setup_graders()

    def setup_graders(self) -> list[dict[str, Any]]:
        """Setup evaluation graders for betting predictions"""
        return [
            {
                "name": "Prediction Accuracy",
                "type": "string_check",
                "operation": "eq",
                "input": "{{ item.model_prediction.predicted_value }}",
                "reference": "{{ item.correct_prediction.predicted_value }}",
            },
            {
                "name": "Confidence Calibration",
                "type": "score_model",
                "model": "gpt-4o-mini",
                "input": [
                    {
                        "role": "system",
                        "content": """
                    Evaluate confidence calibration for betting predictions.

                    Correct prediction: {{ item.correct_prediction }}
                    Model prediction: {{ item.model_prediction }}
                    Actual outcome: {{ item.actual_outcome }}

                    Score from 0-10 where:
                    - 10: Perfect confidence calibration (high confidence = correct, low confidence = wrong)
                    - 5: Random confidence assignment
                    - 0: Inverse confidence (high confidence = wrong, low confidence = correct)
                    """,
                    }
                ],
                "range": [0, 10],
                "pass_threshold": 7,
            },
            {
                "name": "Profit/Loss Performance",
                "type": "score_model",
                "model": "gpt-4o-mini",
                "input": [
                    {
                        "role": "system",
                        "content": """
                    Evaluate the profit/loss performance of betting predictions.

                    P&L: {{ item.profit_loss }}
                    Prediction: {{ item.model_prediction }}

                    Score from 0-10 where:
                    - 10: Highly profitable prediction (>10% ROI)
                    - 7-9: Profitable prediction (0-10% ROI)
                    - 4-6: Break-even prediction (-2% to 2% ROI)
                    - 0-3: Losing prediction (<-2% ROI)
                    """,
                    }
                ],
                "range": [0, 10],
                "pass_threshold": 7,
            },
        ]

    def create_sample_data(self) -> list[dict[str, Any]]:
        """Create sample betting evaluation data"""

        sample_predictions = [
            {
                "prediction_id": "pred_001",
                "correct_prediction": BettingPrediction(
                    game_id="NFL_001",
                    team_home="Patriots",
                    team_away="Bills",
                    prediction_type="spread",
                    predicted_value=-3.5,
                    confidence=0.75,
                    reasoning="Home field advantage, better defense",
                    timestamp="2025-01-01T14:00:00Z",
                ),
                "model_prediction": BettingPrediction(
                    game_id="NFL_001",
                    team_home="Patriots",
                    team_away="Bills",
                    prediction_type="spread",
                    predicted_value=-4.0,
                    confidence=0.8,
                    reasoning="Statistical model favors home team",
                    timestamp="2025-01-01T14:00:00Z",
                ),
                "actual_outcome": {
                    "final_score": "Patriots 24, Bills 17",
                    "spread_result": "Patriots -7",
                },
                "profit_loss": 95.0,  # Won spread bet
            },
            {
                "prediction_id": "pred_002",
                "correct_prediction": BettingPrediction(
                    game_id="NFL_002",
                    team_home="Chiefs",
                    team_away="Broncos",
                    prediction_type="total",
                    predicted_value=45.5,
                    confidence=0.6,
                    reasoning="Weather conditions favor under",
                    timestamp="2025-01-01T16:00:00Z",
                ),
                "model_prediction": BettingPrediction(
                    game_id="NFL_002",
                    team_home="Chiefs",
                    team_away="Broncos",
                    prediction_type="total",
                    predicted_value=48.0,
                    confidence=0.9,
                    reasoning="High-powered offenses suggest over",
                    timestamp="2025-01-01T16:00:00Z",
                ),
                "actual_outcome": {"final_score": "Chiefs 31, Broncos 14", "total_points": 45},
                "profit_loss": -105.0,  # Lost over bet
            },
        ]

        return [
            {"item": record.model_dump()}
            for record in [BettingEvaluationRecord(**pred) for pred in sample_predictions]
        ]


# Initialize evaluation framework
print("🏗️ Setting up EQ12 Evaluation Framework...")
eval_framework = EQ12EvaluationFramework()
sample_data = eval_framework.create_sample_data()

print(f"✅ Framework initialized with {len(eval_framework.graders)} graders")
print(f"📊 Sample data created: {len(sample_data)} evaluation records")

# Display sample data structure
print("\n📝 Sample Evaluation Record:")
print(json.dumps(sample_data[0], indent=2, default=str)[:500] + "...")

In [ ]:
# Business Metrics and ROI Calculation
def calculate_betting_business_metrics(
    evaluation_results: list[dict[str, Any]],
) -> dict[str, float]:
    """Calculate business metrics for EQ12 betting system"""

    total_predictions = len(evaluation_results)
    total_profit_loss = sum(record["item"]["profit_loss"] for record in evaluation_results)

    profitable_predictions = sum(
        1 for record in evaluation_results if record["item"]["profit_loss"] > 0
    )
    accuracy_rate = profitable_predictions / total_predictions if total_predictions > 0 else 0

    # Calculate metrics
    metrics = {
        "total_predictions": total_predictions,
        "accuracy_rate": round(accuracy_rate * 100, 2),
        "total_profit_loss": round(total_profit_loss, 2),
        "avg_profit_per_bet": (
            round(total_profit_loss / total_predictions, 2) if total_predictions > 0 else 0
        ),
        "roi_percentage": round(
            (total_profit_loss / (total_predictions * 100)) * 100, 2
        ),  # Assuming $100 per bet
        "profitable_bet_percentage": (
            round((profitable_predictions / total_predictions) * 100, 2)
            if total_predictions > 0
            else 0
        ),
    }

    # Business impact calculation
    daily_bets = 10  # Estimate
    annual_bets = daily_bets * 365
    annual_profit = metrics["avg_profit_per_bet"] * annual_bets

    metrics.update(
        {
            "projected_annual_profit": round(annual_profit, 2),
            "break_even_accuracy": 52.4,  # Typical sportsbook juice
            "performance_vs_breakeven": round(metrics["accuracy_rate"] - 52.4, 2),
        }
    )

    return metrics


def display_business_dashboard(metrics: dict[str, float]):
    """Display business metrics dashboard"""

    print("📈 EQ12 BETTING SYSTEM DASHBOARD")
    print("=" * 50)
    print(f"🎯 Accuracy Rate: {metrics['accuracy_rate']}%")
    print(f"💰 Total P&L: ${metrics['total_profit_loss']}")
    print(f"📊 Average Profit/Bet: ${metrics['avg_profit_per_bet']}")
    print(f"📈 ROI: {metrics['roi_percentage']}%")
    print(f"✅ Profitable Bets: {metrics['profitable_bet_percentage']}%")
    print(f"🚀 Projected Annual Profit: ${metrics['projected_annual_profit']}")
    print(f"⚖️ vs Break-Even: {metrics['performance_vs_breakeven']:+.1f}%")

    # Performance indicators
    if metrics["roi_percentage"] > 5:
        print("🟢 EXCELLENT: System significantly outperforming market")
    elif metrics["roi_percentage"] > 0:
        print("🟡 GOOD: System profitable, room for improvement")
    else:
        print("🔴 POOR: System losing money, requires optimization")


# Run business metrics calculation
print("💼 Calculating Business Metrics...")
business_metrics = calculate_betting_business_metrics(sample_data)
display_business_dashboard(business_metrics)

# Log business metrics
eq12_config.log_operation(
    "business_metrics_calculation",
    {
        "metrics": business_metrics,
        "timestamp": datetime.utcnow().isoformat(),
        "system_status": "profitable" if business_metrics["roi_percentage"] > 0 else "unprofitable",
    },
)

## Section 5: Multimodal Applications

This section demonstrates combining text, audio, and vision capabilities for comprehensive AI applications. We'll implement examples relevant to EQ12's needs including real-time audio processing and image analysis.

In [ ]:
# Vision + Text: Sports Image Analysis


def analyze_sports_image_with_vision(image_description: str = "screenshot of NFL game statistics"):
    """Demonstrate vision + text analysis for sports content"""

    # Create a simple placeholder for image analysis
    # In real implementation, you would load actual sports images/screenshots

    sample_prompt = f"""
    Analyze this {image_description} and extract key betting-relevant information:

    1. Team names and scores
    2. Game statistics (yards, turnovers, time of possession)
    3. Key player performances
    4. Situational factors (field position, down & distance)
    5. Provide betting insights based on visual data

    Structure your response with clear sections for each analysis point.
    """

    try:
        # Simulated vision analysis (would use actual image in production)
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # Vision-capable model
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert sports analyst specializing in extracting betting insights from visual game data.",
                },
                {"role": "user", "content": sample_prompt},
            ],
            max_tokens=800,
        )

        analysis = response.choices[0].message.content

        print("🖼️ VISION + TEXT ANALYSIS")
        print("=" * 50)
        print(f"Image Type: {image_description}")
        print(f"Analysis:\n{analysis}")

        # Log analysis
        eq12_config.log_operation(
            "vision_text_analysis",
            {
                "image_type": image_description,
                "analysis": analysis,
                "tokens_used": response.usage.total_tokens,
                "modality": "vision_text",
            },
        )

        return analysis

    except Exception as e:
        print(f"❌ Vision analysis error: {e}")
        return None


# Text-to-Speech: Game Commentary Generation
def generate_game_commentary_audio(game_data: dict[str, Any]):
    """Generate audio commentary for game updates"""

    commentary_text = f"""
    Breaking update on the {game_data["home_team"]} versus {game_data["away_team"]} game.
    Current score: {game_data["home_team"]} {game_data["home_score"]},
    {game_data["away_team"]} {game_data["away_score"]}.

    With {game_data["time_remaining"]} remaining in the {game_data["quarter"]} quarter,
    this game is {game_data["betting_status"]} for our betting model predictions.
    """

    try:
        # Generate speech (placeholder - would use actual TTS in production)
        print("🔊 GENERATING GAME COMMENTARY")
        print("=" * 50)
        print(f"Text: {commentary_text}")
        print("Note: Audio generation would be implemented with OpenAI TTS API")

        # Log audio generation
        eq12_config.log_operation(
            "audio_generation",
            {
                "commentary_text": commentary_text,
                "game_data": game_data,
                "modality": "text_to_speech",
            },
        )

        return commentary_text

    except Exception as e:
        print(f"❌ Audio generation error: {e}")
        return None


# Execute multimodal demonstrations
print("🎭 Testing Multimodal Capabilities...")

# Vision + Text Analysis
vision_analysis = analyze_sports_image_with_vision("NFL live game statistics dashboard")

print("\n" + "=" * 70 + "\n")

# Text-to-Speech Commentary
sample_game_data = {
    "home_team": "Patriots",
    "away_team": "Bills",
    "home_score": 14,
    "away_score": 10,
    "quarter": "3rd",
    "time_remaining": "8:45",
    "betting_status": "tracking favorably",
}

commentary = generate_game_commentary_audio(sample_game_data)

## Section 6: Agent Development Workshop

Building intelligent agents using OpenAI's capabilities requires understanding multi-agent collaboration, memory management, and tool orchestration. This section provides hands-on implementation of agentic workflows for EQ12.

In [ ]:
# EQ12 Multi-Agent Betting System
class BettingAgent:
    """Base class for EQ12 betting agents"""

    def __init__(self, name: str, specialization: str, tools: list[dict] | None = None):
        self.name = name
        self.specialization = specialization
        self.tools = tools or []
        self.memory = []
        self.client = client

    def add_memory(self, content: str, metadata: dict | None = None):
        """Add information to agent memory"""
        memory_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "content": content,
            "metadata": metadata or {},
            "agent": self.name,
        }
        self.memory.append(memory_entry)

    def get_recent_memory(self, limit: int = 5) -> list[dict]:
        """Retrieve recent memory entries"""
        return self.memory[-limit:]

    async def analyze(self, prompt: str, context: dict | None = None) -> dict[str, Any]:
        """Base analysis method"""

        # Build context from memory
        memory_context = "\n".join([f"- {mem['content']}" for mem in self.get_recent_memory()])

        full_prompt = f"""
        You are {self.name}, a {self.specialization} for the EQ12 betting system.

        Recent Memory:
        {memory_context}

        Current Analysis Request:
        {prompt}

        Additional Context: {context or "None"}

        Provide analysis specific to your specialization.
        """

        try:
            response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": f"You are {self.name}, specializing in {self.specialization}.",
                    },
                    {"role": "user", "content": full_prompt},
                ],
                tools=self.tools,
            )

            analysis = response.choices[0].message.content

            # Add to memory
            self.add_memory(
                f"Analysis: {prompt[:100]}...",
                {"analysis_result": analysis, "tokens_used": response.usage.total_tokens},
            )

            return {
                "agent": self.name,
                "analysis": analysis,
                "confidence": 0.75,  # Placeholder
                "specialization": self.specialization,
            }

        except Exception as e:
            return {"error": f"Agent {self.name} analysis failed: {e}"}


# Create specialized agents
class StatisticalAgent(BettingAgent):
    def __init__(self):
        super().__init__(
            name="StatBot",
            specialization="Statistical Analysis and Historical Trends",
            tools=[
                {
                    "type": "function",
                    "function": {
                        "name": "query_historical_data",
                        "description": "Query historical game and betting data",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "teams": {"type": "array", "items": {"type": "string"}},
                                "timeframe": {"type": "string"},
                                "stat_type": {"type": "string"},
                            },
                        },
                    },
                }
            ],
        )


class NewsAgent(BettingAgent):
    def __init__(self):
        super().__init__(
            name="NewsBot",
            specialization="News Analysis and Market Sentiment",
            tools=[
                {
                    "type": "function",
                    "function": {
                        "name": "scan_sports_news",
                        "description": "Scan recent sports news for betting insights",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "teams": {"type": "array", "items": {"type": "string"}},
                                "keywords": {"type": "array", "items": {"type": "string"}},
                            },
                        },
                    },
                }
            ],
        )


class RiskAgent(BettingAgent):
    def __init__(self):
        super().__init__(name="RiskBot", specialization="Risk Management and Bankroll Optimization")


# Agent Orchestrator
class EQ12AgentOrchestrator:
    """Coordinates multiple betting agents for comprehensive analysis"""

    def __init__(self):
        self.agents = {"statistical": StatisticalAgent(), "news": NewsAgent(), "risk": RiskAgent()}
        self.collaboration_history = []

    async def coordinate_analysis(self, betting_opportunity: dict[str, Any]) -> dict[str, Any]:
        """Coordinate multi-agent analysis of betting opportunity"""

        print("🤖 MULTI-AGENT ANALYSIS STARTING...")
        print(f"Opportunity: {betting_opportunity['description']}")
        print("=" * 60)

        # Phase 1: Individual agent analysis
        agent_results = {}

        for agent_name, agent in self.agents.items():
            print(f"\n🔍 {agent.name} ({agent.specialization}) analyzing...")

            prompt = f"""
            Analyze this betting opportunity from your specialization perspective:

            Game: {betting_opportunity["game"]}
            Bet Type: {betting_opportunity["bet_type"]}
            Line: {betting_opportunity["line"]}
            Odds: {betting_opportunity["odds"]}

            Provide specific insights relevant to your expertise.
            """

            result = await agent.analyze(prompt, betting_opportunity)
            agent_results[agent_name] = result

            if "error" not in result:
                print(f"✅ {agent.name}: {result['analysis'][:150]}...")
            else:
                print(f"❌ {agent.name}: {result['error']}")

        # Phase 2: Synthesize results
        print("\n🧠 SYNTHESIZING MULTI-AGENT INSIGHTS...")

        synthesis_prompt = f"""
        Synthesize the following agent analyses into a final betting recommendation:

        Statistical Analysis: {agent_results.get("statistical", {}).get("analysis", "N/A")}
        News Analysis: {agent_results.get("news", {}).get("analysis", "N/A")}
        Risk Analysis: {agent_results.get("risk", {}).get("analysis", "N/A")}

        Provide:
        1. Final recommendation (BET/PASS/MONITOR)
        2. Confidence level (0-1)
        3. Reasoning synthesis
        4. Risk assessment
        """

        try:
            synthesis_response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": "You are the EQ12 Master Coordinator synthesizing multi-agent betting analysis.",
                    },
                    {"role": "user", "content": synthesis_prompt},
                ],
            )

            final_recommendation = synthesis_response.choices[0].message.content

            # Log collaboration
            collaboration_record = {
                "timestamp": datetime.utcnow().isoformat(),
                "opportunity": betting_opportunity,
                "agent_results": agent_results,
                "final_recommendation": final_recommendation,
                "coordination_success": True,
            }

            self.collaboration_history.append(collaboration_record)
            eq12_config.log_operation("multi_agent_analysis", collaboration_record)

            print("\n📋 FINAL RECOMMENDATION:")
            print("=" * 60)
            print(final_recommendation)

            return collaboration_record

        except Exception as e:
            print(f"❌ Synthesis error: {e}")
            return {"error": f"Coordination failed: {e}"}


# Initialize and demonstrate multi-agent system
print("🏗️ Initializing EQ12 Multi-Agent Betting System...")
orchestrator = EQ12AgentOrchestrator()

# Sample betting opportunity
sample_opportunity = {
    "description": "NFL Spread Bet Analysis",
    "game": "Kansas City Chiefs @ Buffalo Bills",
    "bet_type": "spread",
    "line": "Chiefs -3.0",
    "odds": -110,
    "stake": 100,
}

print("✅ Multi-agent system ready for demonstration")

In [ ]:
# Execute Multi-Agent Analysis
async def run_multi_agent_demo():
    """Execute the multi-agent betting analysis demonstration"""

    print("🚀 EXECUTING MULTI-AGENT ANALYSIS...")
    result = await orchestrator.coordinate_analysis(sample_opportunity)

    if "error" not in result:
        print("\n📊 COLLABORATION SUMMARY:")
        print(f"Agents Involved: {len(orchestrator.agents)}")
        print(f"Analysis Successful: {result.get('coordination_success', False)}")
        print(f"Total Collaborations: {len(orchestrator.collaboration_history)}")

        # Display agent memory states
        print("\n🧠 AGENT MEMORY STATES:")
        for _name, agent in orchestrator.agents.items():
            memory_count = len(agent.memory)
            print(f"{agent.name}: {memory_count} memory entries")

    return result


# Run the demonstration
multi_agent_result = await run_multi_agent_demo()

## Section 7: Production Deployment Guide

This section implements production-ready patterns including monitoring, security, rate limiting, and cost optimization based on OpenAI Cookbook best practices.

In [ ]:
# Production Monitoring and Rate Limiting
from collections import deque
from collections.abc import AsyncGenerator


class EQ12ProductionMonitor:
    """Production monitoring system for EQ12 OpenAI integration"""

    def __init__(self):
        self.request_log = deque(maxlen=10000)  # Last 10k requests
        self.rate_limits = {
            "requests_per_minute": 60,
            "tokens_per_minute": 150000,
            "daily_cost_limit": 100.0,
        }
        self.current_usage = {
            "requests_this_minute": 0,
            "tokens_this_minute": 0,
            "daily_cost": 0.0,
            "last_reset": time.time(),
        }
        self.alerts = []

    def check_rate_limits(self) -> dict[str, bool]:
        """Check if current usage is within rate limits"""

        current_time = time.time()

        # Reset counters if a minute has passed
        if current_time - self.current_usage["last_reset"] >= 60:
            self.current_usage["requests_this_minute"] = 0
            self.current_usage["tokens_this_minute"] = 0
            self.current_usage["last_reset"] = current_time

        # Check limits
        limits_ok = {
            "requests": self.current_usage["requests_this_minute"]
            < self.rate_limits["requests_per_minute"],
            "tokens": self.current_usage["tokens_this_minute"]
            < self.rate_limits["tokens_per_minute"],
            "cost": self.current_usage["daily_cost"] < self.rate_limits["daily_cost_limit"],
        }

        return limits_ok

    def log_request(self, request_data: dict[str, Any]):
        """Log an API request with monitoring data"""

        timestamp = datetime.utcnow().isoformat()

        log_entry = {
            "timestamp": timestamp,
            "model": request_data.get("model"),
            "tokens": request_data.get("tokens", 0),
            "cost": request_data.get("cost", 0.0),
            "latency": request_data.get("latency", 0.0),
            "success": request_data.get("success", True),
            "error": request_data.get("error"),
        }

        # Update current usage
        self.current_usage["requests_this_minute"] += 1
        self.current_usage["tokens_this_minute"] += log_entry["tokens"]
        self.current_usage["daily_cost"] += log_entry["cost"]

        # Log to deque and file
        self.request_log.append(log_entry)
        eq12_config.log_operation("api_request", log_entry)

        # Check for alerts
        self._check_alerts(log_entry)

    def _check_alerts(self, log_entry: dict[str, Any]):
        """Check for alerting conditions"""

        # High latency alert
        if log_entry["latency"] > 5.0:
            alert = {
                "type": "high_latency",
                "message": f"High latency detected: {log_entry['latency']:.2f}s",
                "timestamp": log_entry["timestamp"],
            }
            self.alerts.append(alert)
            print(f"🚨 ALERT: {alert['message']}")

        # Cost threshold alert
        if self.current_usage["daily_cost"] > self.rate_limits["daily_cost_limit"] * 0.8:
            alert = {
                "type": "cost_threshold",
                "message": f"Daily cost approaching limit: ${self.current_usage['daily_cost']:.2f}",
                "timestamp": log_entry["timestamp"],
            }
            self.alerts.append(alert)
            print(f"💰 ALERT: {alert['message']}")

    def get_performance_metrics(self) -> dict[str, Any]:
        """Calculate performance metrics from recent requests"""

        if not self.request_log:
            return {"error": "No request data available"}

        recent_requests = list(self.request_log)
        successful_requests = [r for r in recent_requests if r["success"]]

        metrics = {
            "total_requests": len(recent_requests),
            "successful_requests": len(successful_requests),
            "success_rate": len(successful_requests) / len(recent_requests) * 100,
            "avg_latency": (
                sum(r["latency"] for r in successful_requests) / len(successful_requests)
                if successful_requests
                else 0
            ),
            "total_tokens": sum(r["tokens"] for r in recent_requests),
            "total_cost": sum(r["cost"] for r in recent_requests),
            "current_usage": self.current_usage.copy(),
            "rate_limits": self.rate_limits.copy(),
            "alerts_count": len(self.alerts),
        }

        return metrics


# Security and Authentication Layer
class EQ12SecurityManager:
    """Security manager for EQ12 production deployment"""

    def __init__(self):
        self.api_keys = self._load_api_keys()
        self.request_signatures = {}

    def _load_api_keys(self) -> dict[str, str]:
        """Load API keys securely from environment"""
        return {
            "openai": os.getenv("OPENAI_API_KEY"),
            "odds_api": os.getenv("ODDS_API_KEY", "demo_key"),
            "telegram": os.getenv("TELEGRAM_BOT_TOKEN", "demo_token"),
        }

    def validate_request(self, request_data: dict[str, Any]) -> bool:
        """Validate incoming requests for security"""

        # Check for required fields
        required_fields = ["user_id", "request_type", "timestamp"]
        for field in required_fields:
            if field not in request_data:
                print(f"🔒 Security: Missing required field: {field}")
                return False

        # Check timestamp freshness (within 5 minutes)
        try:
            request_time = datetime.fromisoformat(request_data["timestamp"])
            current_time = datetime.utcnow()

            if (current_time - request_time).total_seconds() > 300:
                print("🔒 Security: Request timestamp too old")
                return False
        except:
            print("🔒 Security: Invalid timestamp format")
            return False

        # Log security validation
        eq12_config.log_operation(
            "security_validation",
            {
                "user_id": request_data["user_id"],
                "request_type": request_data["request_type"],
                "validation_result": "approved",
            },
        )

        return True

    def encrypt_sensitive_data(self, data: str) -> str:
        """Encrypt sensitive data (placeholder implementation)"""
        # In production, use proper encryption (e.g., Fernet, AES)
        return f"ENCRYPTED_{hash(data) % 10000}"

    def get_security_status(self) -> dict[str, Any]:
        """Get current security status"""
        return {
            "api_keys_configured": len(
                [k for k in self.api_keys.values() if k and k != "demo_key"]
            ),
            "encryption_enabled": True,
            "request_validation": True,
            "last_security_check": datetime.utcnow().isoformat(),
        }


# Initialize production components
print("🏭 Initializing Production Systems...")
monitor = EQ12ProductionMonitor()
security = EQ12SecurityManager()

print("✅ Production monitoring system initialized")
print("✅ Security manager initialized")
print(f"🔐 Security status: {security.get_security_status()}")

# Demonstrate monitoring
sample_request = {
    "model": "gpt-4o-mini",
    "tokens": 500,
    "cost": 0.025,
    "latency": 1.2,
    "success": True,
}

monitor.log_request(sample_request)
metrics = monitor.get_performance_metrics()

print("\n📊 Performance Metrics:")
for key, value in metrics.items():
    if key not in ["current_usage", "rate_limits"]:
        print(f"{key}: {value}")

In [ ]:
# Cost Optimization and Resource Management
class EQ12CostOptimizer:
    """Cost optimization strategies for EQ12 OpenAI usage"""

    def __init__(self):
        self.model_costs = {
            "gpt-4o-mini": {"input": 0.000150, "output": 0.000600},
            "gpt-4o": {"input": 0.0025, "output": 0.01},
            "gpt-3.5-turbo": {"input": 0.0005, "output": 0.0015},
        }
        self.usage_history = []
        self.optimization_rules = {
            "use_mini_for_simple": True,
            "batch_similar_requests": True,
            "cache_frequent_queries": True,
            "optimize_prompt_length": True,
        }

    def suggest_optimal_model(self, task_complexity: str, max_tokens: int) -> str:
        """Suggest most cost-effective model for given task"""

        # Simple classification tasks -> use mini
        if task_complexity in ["simple", "classification", "extraction"]:
            return "gpt-4o-mini"

        # Complex reasoning or long responses -> use full model
        if task_complexity in ["complex", "reasoning", "creative"]:
            if max_tokens > 2000:
                return "gpt-4o"
            return "gpt-4o-mini"

        # Default to mini for cost optimization
        return "gpt-4o-mini"

    def calculate_request_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost for a specific request"""

        if model not in self.model_costs:
            model = "gpt-4o-mini"  # Default fallback

        costs = self.model_costs[model]
        total_cost = (input_tokens * costs["input"] + output_tokens * costs["output"]) / 1000

        return total_cost

    def optimize_prompt(self, prompt: str) -> Tuple[str, dict[str, Any]]:
        """Optimize prompt for cost efficiency while maintaining effectiveness"""

        original_length = len(prompt)
        optimizations_applied = []

        # Remove excessive whitespace
        optimized = " ".join(prompt.split())
        if len(optimized) < original_length:
            optimizations_applied.append("whitespace_cleanup")

        # Shorten verbose instructions while keeping clarity
        replacements = {
            "Please provide a detailed analysis of": "Analyze:",
            "I would like you to": "",
            "Could you please": "",
            "It would be great if": "",
        }

        for old, new in replacements.items():
            if old in optimized:
                optimized = optimized.replace(old, new)
                optimizations_applied.append("shortened_instruction")

        optimization_info = {
            "original_length": original_length,
            "optimized_length": len(optimized),
            "reduction_percent": (1 - len(optimized) / original_length) * 100,
            "optimizations": optimizations_applied,
        }

        return optimized, optimization_info

    def get_usage_report(self) -> dict[str, Any]:
        """Generate cost usage report and recommendations"""

        if not self.usage_history:
            return {"message": "No usage data available yet"}

        total_requests = len(self.usage_history)
        total_cost = sum(req.get("cost", 0) for req in self.usage_history)
        avg_cost_per_request = total_cost / total_requests if total_requests > 0 else 0

        # Model usage breakdown
        model_usage = {}
        for req in self.usage_history:
            model = req.get("model", "unknown")
            if model not in model_usage:
                model_usage[model] = {"requests": 0, "cost": 0}
            model_usage[model]["requests"] += 1
            model_usage[model]["cost"] += req.get("cost", 0)

        # Generate recommendations
        recommendations = []

        # Check if using expensive models for simple tasks
        gpt4_usage = model_usage.get("gpt-4o", {}).get("requests", 0)
        mini_usage = model_usage.get("gpt-4o-mini", {}).get("requests", 0)

        if gpt4_usage > mini_usage:
            recommendations.append(
                "Consider using gpt-4o-mini for simpler classification and extraction tasks"
            )

        if avg_cost_per_request > 0.05:
            recommendations.append(
                "Average cost per request is high - consider prompt optimization"
            )

        return {
            "total_requests": total_requests,
            "total_cost": total_cost,
            "avg_cost_per_request": avg_cost_per_request,
            "model_breakdown": model_usage,
            "recommendations": recommendations,
            "potential_monthly_cost": total_cost * 30 if total_requests > 0 else 0,
        }


# Deployment Configuration Management
class EQ12DeploymentConfig:
    """Manage deployment configurations for different environments"""

    def __init__(self):
        self.environments = {
            "development": {
                "rate_limits": {"requests_per_minute": 10, "daily_cost_limit": 5.0},
                "models_allowed": ["gpt-4o-mini"],
                "logging_level": "DEBUG",
                "monitoring_enabled": True,
            },
            "staging": {
                "rate_limits": {"requests_per_minute": 30, "daily_cost_limit": 25.0},
                "models_allowed": ["gpt-4o-mini", "gpt-4o"],
                "logging_level": "INFO",
                "monitoring_enabled": True,
            },
            "production": {
                "rate_limits": {"requests_per_minute": 60, "daily_cost_limit": 100.0},
                "models_allowed": ["gpt-4o-mini", "gpt-4o"],
                "logging_level": "WARNING",
                "monitoring_enabled": True,
            },
        }
        self.current_env = "development"

    def set_environment(self, env_name: str):
        """Set current deployment environment"""
        if env_name in self.environments:
            self.current_env = env_name
            print(f"🏗️ Environment set to: {env_name}")
            return True
        print(f"❌ Unknown environment: {env_name}")
        return False

    def get_current_config(self) -> dict[str, Any]:
        """Get configuration for current environment"""
        return self.environments.get(self.current_env, {})

    def validate_deployment(self) -> dict[str, bool]:
        """Validate current deployment configuration"""

        config = self.get_current_config()
        validation = {
            "rate_limits_configured": "rate_limits" in config,
            "models_restricted": len(config.get("models_allowed", [])) > 0,
            "logging_configured": "logging_level" in config,
            "monitoring_enabled": config.get("monitoring_enabled", False),
        }

        all_valid = all(validation.values())
        print(f"🔍 Deployment validation: {'✅ PASSED' if all_valid else '❌ FAILED'}")

        return validation


# Initialize cost optimization and deployment management
print("💰 Initializing Cost Optimization...")
cost_optimizer = EQ12CostOptimizer()
deployment_config = EQ12DeploymentConfig()

print("✅ Cost optimizer initialized")
print("✅ Deployment config manager initialized")

# Demonstrate cost optimization
sample_prompt = "Please provide a very detailed and comprehensive analysis of the following betting scenario with extensive background information and contextual details..."

optimized_prompt, opt_info = cost_optimizer.optimize_prompt(sample_prompt)

print("\n📝 Prompt Optimization Demo:")
print(f"Original length: {opt_info['original_length']} characters")
print(f"Optimized length: {opt_info['optimized_length']} characters")
print(f"Reduction: {opt_info['reduction_percent']:.1f}%")
print(f"Optimizations applied: {opt_info['optimizations']}")

# Demonstrate deployment configuration
deployment_config.set_environment("production")
prod_config = deployment_config.get_current_config()
validation_result = deployment_config.validate_deployment()

print("\n🚀 Production Configuration:")
print(f"Rate limits: {prod_config['rate_limits']}")
print(f"Allowed models: {prod_config['models_allowed']}")
print(f"Deployment validation: {validation_result}")

## Section 8: Advanced Integration Examples

This section demonstrates advanced integration patterns combining multiple OpenAI features for sophisticated EQ12 betting automation workflows.

In [ ]:
# Advanced Multi-Modal Betting Analysis System
class EQ12AdvancedAnalyzer:
    """Advanced betting analyzer combining text, vision, and structured data"""

    def __init__(self):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.analysis_history = []

    async def analyze_game_with_context(
        self,
        game_data: dict[str, Any],
        news_articles: list[str] | None = None,
        injury_reports_image: str | None = None,
        weather_data: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        """Comprehensive game analysis using multiple data sources"""

        # Structure the betting analysis request
        analysis_request = BettingPrediction(
            game_id=game_data.get("game_id", "unknown"),
            teams=f"{game_data.get('home_team')} vs {game_data.get('away_team')}",
            prediction_type="comprehensive_analysis",
            confidence_level=0.0,  # Will be determined by analysis
            reasoning="Multi-modal analysis pending",
        )

        # Prepare multi-modal analysis
        messages = [
            {
                "role": "system",
                "content": """You are an expert sports betting analyst for EQ12.
                Provide comprehensive analysis using all available data sources.
                Consider team statistics, news sentiment, injury reports, and environmental factors.
                Provide specific confidence levels and actionable betting recommendations.""",
            },
            {
                "role": "user",
                "content": f"""Analyze this upcoming game:

                GAME DATA:
                {json.dumps(game_data, indent=2)}

                WEATHER CONDITIONS:
                {json.dumps(weather_data, indent=2) if weather_data else "No weather data available"}
                """,
            },
        ]

        # Add news analysis if available
        if news_articles:
            news_summary = "\n".join(news_articles[:3])  # Limit to top 3 articles
            messages.append(
                {
                    "role": "user",
                    "content": f"""RECENT NEWS ANALYSIS:
                {news_summary}

                How do these news items affect the betting outlook?""",
                }
            )

        # Add injury report image analysis if available
        if injury_reports_image:
            messages.append(
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Analyze this injury report image and factor into betting recommendations:",
                        },
                        {"type": "image_url", "image_url": {"url": injury_reports_image}},
                    ],
                }
            )

        try:
            # Use GPT-4o for comprehensive analysis
            response = await self.client.chat.completions.acreate(
                model="gpt-4o",
                messages=messages,
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "comprehensive_betting_analysis",
                        "schema": {
                            "type": "object",
                            "properties": {
                                "overall_confidence": {
                                    "type": "number",
                                    "minimum": 0,
                                    "maximum": 1,
                                },
                                "recommended_bets": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "bet_type": {"type": "string"},
                                            "selection": {"type": "string"},
                                            "confidence": {"type": "number"},
                                            "reasoning": {"type": "string"},
                                            "risk_level": {
                                                "type": "string",
                                                "enum": ["low", "medium", "high"],
                                            },
                                        },
                                        "required": [
                                            "bet_type",
                                            "selection",
                                            "confidence",
                                            "reasoning",
                                            "risk_level",
                                        ],
                                    },
                                },
                                "key_factors": {"type": "array", "items": {"type": "string"}},
                                "risk_assessment": {"type": "string"},
                                "bankroll_recommendation": {"type": "string"},
                            },
                            "required": [
                                "overall_confidence",
                                "recommended_bets",
                                "key_factors",
                                "risk_assessment",
                            ],
                        },
                    },
                },
                temperature=0.3,
            )

            analysis_result = json.loads(response.choices[0].message.content)

            # Update our prediction with the analysis
            analysis_request.confidence_level = analysis_result["overall_confidence"]
            analysis_request.reasoning = (
                f"Multi-modal analysis: {analysis_result['risk_assessment']}"
            )

            # Comprehensive result
            final_result = {
                "prediction": analysis_request.dict(),
                "detailed_analysis": analysis_result,
                "data_sources_used": {
                    "game_statistics": bool(game_data),
                    "news_analysis": bool(news_articles),
                    "injury_reports": bool(injury_reports_image),
                    "weather_data": bool(weather_data),
                },
                "timestamp": datetime.utcnow().isoformat(),
            }

            # Store for learning
            self.analysis_history.append(final_result)

            # Log the comprehensive analysis
            eq12_config.log_operation("advanced_betting_analysis", final_result)

            return final_result

        except Exception as e:
            error_result = {
                "error": str(e),
                "prediction": analysis_request.dict(),
                "timestamp": datetime.utcnow().isoformat(),
            }

            eq12_config.log_operation("analysis_error", error_result)
            return error_result


# Real-time Betting Pipeline with Streaming
class EQ12StreamingPipeline:
    """Real-time betting analysis pipeline with streaming responses"""

    def __init__(self):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.active_streams = {}

    async def stream_live_analysis(self, game_id: str) -> AsyncGenerator[dict[str, Any], None]:
        """Stream live betting analysis updates"""

        system_prompt = f"""You are providing live betting analysis for game {game_id}.
        Stream continuous updates as new information becomes available.
        Focus on actionable insights and changing odds opportunities.
        Each update should include confidence levels and specific recommendations."""

        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Begin live analysis for game {game_id}. Provide streaming updates.",
            },
        ]

        try:
            stream = await self.client.chat.completions.acreate(
                model="gpt-4o-mini",  # Use mini for cost-effective streaming
                messages=messages,
                stream=True,
                temperature=0.4,
                max_tokens=500,
            )

            current_analysis = ""

            async for chunk in stream:
                if chunk.choices[0].delta.content is not None:
                    content_chunk = chunk.choices[0].delta.content
                    current_analysis += content_chunk

                    # Yield incremental updates
                    yield {
                        "game_id": game_id,
                        "type": "streaming_update",
                        "content_chunk": content_chunk,
                        "cumulative_content": current_analysis,
                        "timestamp": datetime.utcnow().isoformat(),
                    }

            # Final consolidated analysis
            yield {
                "game_id": game_id,
                "type": "final_analysis",
                "complete_analysis": current_analysis,
                "timestamp": datetime.utcnow().isoformat(),
            }

        except Exception as e:
            yield {
                "game_id": game_id,
                "type": "error",
                "error": str(e),
                "timestamp": datetime.utcnow().isoformat(),
            }

    async def process_live_odds_changes(self, odds_update: dict[str, Any]) -> dict[str, Any]:
        """Process real-time odds changes with immediate analysis"""

        # Quick analysis of odds movement
        messages = [
            {
                "role": "system",
                "content": """Analyze odds movement for immediate betting opportunities.
                Focus on significant line movements and value identification.
                Provide fast, actionable recommendations.""",
            },
            {
                "role": "user",
                "content": f"""ODDS UPDATE:
                {json.dumps(odds_update, indent=2)}

                Provide immediate analysis of this odds movement.""",
            },
        ]

        try:
            response = await self.client.chat.completions.acreate(
                model="gpt-4o-mini", messages=messages, temperature=0.2, max_tokens=300
            )

            analysis = {
                "odds_update": odds_update,
                "analysis": response.choices[0].message.content,
                "processing_time": datetime.utcnow().isoformat(),
                "recommendation_urgency": self._assess_urgency(odds_update),
            }

            eq12_config.log_operation("live_odds_analysis", analysis)

            return analysis

        except Exception as e:
            return {
                "odds_update": odds_update,
                "error": str(e),
                "timestamp": datetime.utcnow().isoformat(),
            }

    def _assess_urgency(self, odds_update: dict[str, Any]) -> str:
        """Assess urgency level of odds movement"""

        # Simple heuristic for urgency assessment
        movement_size = abs(odds_update.get("movement_percentage", 0))

        if movement_size > 10:
            return "high"
        if movement_size > 5:
            return "medium"
        return "low"


# Initialize advanced systems
print("🚀 Initializing Advanced Integration Systems...")
advanced_analyzer = EQ12AdvancedAnalyzer()
streaming_pipeline = EQ12StreamingPipeline()

print("✅ Advanced multi-modal analyzer initialized")
print("✅ Real-time streaming pipeline initialized")

# Demonstrate advanced analysis
sample_game_data = {
    "game_id": "NBA_2024_001",
    "home_team": "Los Angeles Lakers",
    "away_team": "Boston Celtics",
    "date": "2024-03-15",
    "current_odds": {"home": -110, "away": +105},
    "home_record": "45-20",
    "away_record": "43-22",
}

sample_news = [
    "Lakers star player questionable with ankle injury",
    "Celtics on 5-game winning streak heading into matchup",
    "Weather conditions clear for tonight's game",
]

sample_weather = {"temperature": 72, "humidity": 45, "wind": "5 mph", "conditions": "clear"}

print("\n🔍 Sample Advanced Analysis:")
print(f"Analyzing: {sample_game_data['home_team']} vs {sample_game_data['away_team']}")
print(f"Data sources: Game stats, {len(sample_news)} news articles, weather data")
print("📊 (In production, this would generate comprehensive multi-modal analysis)")

## Summary and Next Steps

Congratulations! You've now explored the comprehensive OpenAI Cookbook implementation for EQ12 sports betting automation. Here's what we've covered and how to proceed:

### What You've Learned

1. **GPT-5 Advanced Features**: Verbosity control, reasoning effort, and context-free grammars
2. **API Integration Patterns**: Async/await, error handling, and structured outputs  
3. **Evaluation Frameworks**: Custom graders, business metrics, and performance tracking
4. **Multi-Agent Systems**: Specialized betting agents with coordination patterns
5. **Production Deployment**: Monitoring, security, cost optimization, and scaling strategies
6. **Advanced Integrations**: Multi-modal analysis and real-time streaming pipelines

### EQ12 Master Profile Integration

To integrate these capabilities into your EQ12 Master Profile shortcuts:

1. **Copy Key Functions**: Extract the configuration classes and core functions to your main EQ12 scripts
2. **Update PowerShell Wrappers**: Add new shortcuts that call the OpenAI-powered analysis functions
3. **Configure Environment**: Set up the required API keys and logging directories
4. **Test Integration**: Run the evaluation frameworks to ensure everything works correctly

### Immediate Action Items

- [ ] Set up OpenAI API keys in your environment variables
- [ ] Configure the EQ12Config logging system in your main scripts  
- [ ] Test the basic GPT-5 features with simple betting predictions
- [ ] Implement the monitoring system for production usage
- [ ] Integrate multi-agent analysis into your existing betting workflows

### Production Checklist

Before deploying to production:
- ✅ API rate limiting configured
- ✅ Security validation implemented  
- ✅ Cost monitoring enabled
- ✅ Error handling and logging in place
- ✅ Performance metrics tracking
- ✅ Evaluation frameworks running

This notebook serves as your comprehensive reference for building sophisticated AI-powered betting automation with the latest OpenAI capabilities. Keep experimenting and building!